In [8]:
from pathlib import Path
import py_compile
import shutil

script_path = Path(
    r"C:\Users\patri\vrp_project\notebooks\vrp_repaired_wilder_rsi_signal_base_v1.py"
)

if not script_path.exists():
    raise FileNotFoundError(script_path)

text = script_path.read_text(encoding="utf-8")

backup_path = script_path.with_name(
    "vrp_repaired_wilder_rsi_signal_base_v1_BACKUP_1_0_1.py"
)
shutil.copy2(script_path, backup_path)

old_version = 'SCRIPT_VERSION = "1.0.1"'
new_version = 'SCRIPT_VERSION = "1.0.2"'

old_imports = """import hashlib
import json
import os
"""

new_imports = """import hashlib
import json
import re
import os
"""

old_function = '''def is_legacy_rsi_column(column: str) -> bool:
    return "rsi" in column.lower()
'''

new_function = '''def is_legacy_rsi_column(column: str) -> bool:
    """Identify actual RSI fields without matching unrelated words like ``version``.

    Examples matched: RSI, RSI14, rsi14_final, old_rsi14, rsi_formula_version.
    Examples not matched: final_signal_version, model_version.
    """
    normalized = re.sub(r"[^a-z0-9]+", "_", str(column).lower()).strip("_")
    tokens = normalized.split("_") if normalized else []
    return any(
        token in {"rsi", "rsi14"} or re.fullmatch(r"rsi\\d+", token)
        for token in tokens
    )
'''

for expected, label in [
    (old_version, "v1.0.1 version line"),
    (old_imports, "import block"),
    (old_function, "old RSI-column filter"),
]:
    if expected not in text:
        raise RuntimeError(f"Could not find expected {label}; script was not modified.")

text = text.replace(old_version, new_version, 1)
text = text.replace(old_imports, new_imports, 1)
text = text.replace(old_function, new_function, 1)

script_path.write_text(text, encoding="utf-8")

py_compile.compile(str(script_path), doraise=True)

verified = script_path.read_text(encoding="utf-8")

assert 'SCRIPT_VERSION = "1.0.2"' in verified
assert 'return "rsi" in column.lower()' not in verified
assert "import re" in verified

print("Backup:", backup_path)
print("Patched:", script_path)
print('Verified: SCRIPT_VERSION = "1.0.2"')
print("Compile: PASS")

Backup: C:\Users\patri\vrp_project\notebooks\vrp_repaired_wilder_rsi_signal_base_v1_BACKUP_1_0_1.py
Patched: C:\Users\patri\vrp_project\notebooks\vrp_repaired_wilder_rsi_signal_base_v1.py
Verified: SCRIPT_VERSION = "1.0.2"
Compile: PASS


In [9]:
%cd "C:\Users\patri\vrp_project\notebooks"
!py -u ".\vrp_repaired_wilder_rsi_signal_base_v1.py" --project-root "C:\Users\patri\vrp_project"

C:\Users\patri\vrp_project\notebooks

Step 1 — Repaired Wilder RSI signal base
Script:             vrp_repaired_wilder_rsi_signal_base_v1.py v1.0.2
Project root:       C:\Users\patri\vrp_project
Signal input:       C:\Users\patri\vrp_project\data\processed\vrp_final_signal\vrp_final_corsi_signal_base_panel_v1.parquet
RSI input:          C:\Users\patri\vrp_project\data\processed\market_data\spy_wilder_rsi14_history_v1.parquet
Base output:        C:\Users\patri\vrp_project\data\processed\vrp_repaired_wilder_rsi_signal\vrp_repaired_wilder_rsi_signal_base_v1.parquet
Snapshot output:    C:\Users\patri\vrp_project\data\processed\vrp_repaired_wilder_rsi_signal\vrp_repaired_wilder_rsi_latest_snapshot_v1.parquet
Audit directory:    C:\Users\patri\vrp_project\data\audit\repaired_wilder_rsi_signal
Allowed RSI version(s): ['wilder_rsi14_spy_close_v2_long_warmup']

Loading inputs
Signal rows/columns: 14,724 / 36
RSI rows/columns:    2,140 / 11
Resolved signal columns:
  trade_date           -> trad

In [14]:
%cd "C:\Users\patri\vrp_project\notebooks"
!py -u ".\vrp_wilder_rsi_trade_outcome_source_inventory_v1.py" --project-root "C:\Users\patri\vrp_project"

C:\Users\patri\vrp_project\notebooks

Trade outcome source inventory — read only
Script:             vrp_wilder_rsi_trade_outcome_source_inventory_v1.py v1.0.0
Project root:       C:\Users\patri\vrp_project
Scan root 1:        C:\Users\patri\vrp_project\data\processed
Scan root 2:        C:\Users\patri\vrp_project\data\audit
Audit directory:    C:\Users\patri\vrp_project\data\audit\wilder_rsi_trade_outcome_source_inventory
Target tenors:      [12, 15, 18, 21, 24, 27, 30, 33]
9D status:          inventory only; remains inactive for optimization

Inspecting file schemas
Inspected 250 / 3,162 files...
Inspected 500 / 3,162 files...
Inspected 750 / 3,162 files...
Inspected 1,000 / 3,162 files...
Inspected 1,250 / 3,162 files...
Inspected 1,500 / 3,162 files...
Inspected 1,750 / 3,162 files...
Inspected 2,000 / 3,162 files...
Inspected 2,250 / 3,162 files...
Inspected 2,500 / 3,162 files...
Inspected 2,750 / 3,162 files...
Inspected 3,000 / 3,162 files...
Files scanned:              3,162
P

In [15]:
%cd "C:\Users\patri\vrp_project\notebooks"
!py -u ".\vrp_wilder_rsi_tenor_isolated_parameter_sweep_v1.py" --project-root "C:\Users\patri\vrp_project"

C:\Users\patri\vrp_project\notebooks

Tenor-isolated repaired-Wilder-RSI parameter sweep
Script:             vrp_wilder_rsi_tenor_isolated_parameter_sweep_v1.py v1.0.0
Project root:       C:\Users\patri\vrp_project
Signal input:       C:\Users\patri\vrp_project\data\processed\vrp_repaired_wilder_rsi_signal\vrp_repaired_wilder_rsi_signal_base_v1.parquet
Outcome input:      C:\Users\patri\vrp_project\data\processed\naked_atm_put_tenor_sizing_trades_v0_1.parquet
Results output:     C:\Users\patri\vrp_project\data\processed\vrp_wilder_rsi_parameter_sweep\vrp_wilder_rsi_tenor_isolated_parameter_sweep_results_v1.parquet
Leaderboard output: C:\Users\patri\vrp_project\data\processed\vrp_wilder_rsi_parameter_sweep\vrp_wilder_rsi_tenor_isolated_parameter_leaderboard_v1.parquet
Audit directory:    C:\Users\patri\vrp_project\data\audit\wilder_rsi_tenor_isolated_parameter_sweep
Target tenors:      [12, 15, 18, 21, 24, 27, 30, 33]
Selection rule:     none; every qualifying tenor is counted independe

In [16]:
from pathlib import Path
import subprocess

project_root = Path(r"C:\Users\patri\vrp_project")

matches = sorted(
    (project_root / "data" / "processed").rglob(
        "naked_atm_put_tenor_sizing_trades_v0_1.parquet"
    )
)

print("Matching outcome files:")
for path in matches:
    print(" -", path)

if len(matches) == 0:
    raise FileNotFoundError(
        "naked_atm_put_tenor_sizing_trades_v0_1.parquet was not found "
        "anywhere under data\\processed."
    )

if len(matches) > 1:
    raise RuntimeError(
        "More than one exact outcome file was found. "
        "Do not choose one automatically."
    )

outcome_path = matches[0]

cmd = [
    "py",
    "-u",
    str(
        project_root
        / "notebooks"
        / "vrp_wilder_rsi_tenor_isolated_parameter_sweep_v1.py"
    ),
    "--project-root",
    str(project_root),
    "--outcome-path",
    str(outcome_path),
]

print("\nUsing outcome source:")
print(outcome_path)

subprocess.run(cmd, check=True)

Matching outcome files:
 - C:\Users\patri\vrp_project\data\processed\old\naked_atm_put_tenor_sizing_trades_v0_1.parquet

Using outcome source:
C:\Users\patri\vrp_project\data\processed\old\naked_atm_put_tenor_sizing_trades_v0_1.parquet


CalledProcessError: Command '['py', '-u', 'C:\\Users\\patri\\vrp_project\\notebooks\\vrp_wilder_rsi_tenor_isolated_parameter_sweep_v1.py', '--project-root', 'C:\\Users\\patri\\vrp_project', '--outcome-path', 'C:\\Users\\patri\\vrp_project\\data\\processed\\old\\naked_atm_put_tenor_sizing_trades_v0_1.parquet']' returned non-zero exit status 1.